In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path

CLEAN_DATA_DIR = Path("../clean_data")

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet(f"{CLEAN_DATA_DIR}/4_eea_co2_emissions_from_passenger_cars-001.parquet")
df.show(5)

In [ ]:
from pyspark.sql import functions as F

num_cols = [
    "mass_in_running_order (kg)", "co2_emissions_WLTP (g/km)", 
    "engine_capacity (cm3)", "engine_power (KW)", "electric_energy_consumption (Wh/km)"
]
str_cols = ["member_state", "make", "commercial_name", "fuel_type", "fuel_mode"]
all_cols = str_cols + num_cols

def count_missing(col_name):
    is_missing = F.col(col_name).isNull() | (F.trim(F.col(col_name)) == "")
    return F.sum(is_missing.cast("int")).alias(f"{col_name}_nulls")

def count_zeros(col_name):
    is_zero = F.col(col_name) == 0
    return F.sum(is_zero.cast("int")).alias(f"{col_name}_zeros")

zeros = [count_missing(c) for c in all_cols] + [count_zeros(c) for c in num_cols]

df_zeros = df.groupBy("year").agg(*zeros).orderBy("year")
df_zeros.show(truncate=False)